# BERT — encoder-only Transformer with masked-LM pretraining

> Tutorial pair for [`bert.py`](bert.py). Read
> [`transformer.ipynb`](transformer.ipynb) and
> [`attention.ipynb`](../attention/attention.ipynb) first.

## 1. Intuition
GPT reads left-to-right; BERT reads **both directions at once**. It keeps only the
Transformer *encoder* (no causal mask), so every token's representation is built
from the entire sentence. You cannot train such a model with next-token
prediction (it would trivially see the answer), so BERT instead **masks ~15% of
the tokens** and learns to fill them back in — the Masked Language Model objective.
A special **[CLS]** token aggregates the sequence for classification.

## 2. Concept (the slide)
- **Bidirectional self-attention:** no causal mask; only a *padding* mask hides
  [PAD] positions.
- **Three embeddings summed:** token + learned positional + **segment** (sentence
  A vs B), enabling sentence-pair inputs `[CLS] A [SEP] B [SEP]`.
- **MLM head:** predict the original id at every masked position (tied to the input
  embedding matrix).
- **NSP head (conceptual):** classify from the pooled [CLS] vector whether
  sentence B follows A — uses the segment embeddings.

## 3. Math derivation

### 3.1 Bidirectional attention
A BERT layer is exactly scaled-dot-product self-attention **without** a causal
mask:
$$A=\operatorname{softmax}\!\Big(\tfrac{QK^\top}{\sqrt{d_k}}+M_{\text{pad}}\Big)V,$$
where $M_{\text{pad}}[i,j]=-\infty$ iff key $j$ is a [PAD] token, else $0$. Because
nothing blocks the upper triangle, token $i$'s output depends on the **whole**
sequence — left and right context.

### 3.2 The masked-language-model objective
Let $\mathcal{M}\subset\{1,\dots,L\}$ be the randomly chosen masked positions and
$\hat x$ the corrupted input. BERT maximizes the log-probability of the *original*
tokens at those positions:
$$\mathcal{L}_{\text{MLM}}
=-\sum_{t\in\mathcal{M}}\log p_\theta\!\big(x_t \mid \hat x\big).$$
Only the masked positions contribute (others have label $-100$ and are ignored).
This is a denoising autoencoder over discrete tokens.

### 3.3 The 80/10/10 masking scheme
For each chosen position $t\in\mathcal{M}$:
$$\hat x_t=\begin{cases}
\texttt{[MASK]} & \text{with prob } 0.8\\
\text{random token} & \text{with prob } 0.1\\
x_t\ \text{(unchanged)} & \text{with prob } 0.1.
\end{cases}$$
Why not always `[MASK]`? Because `[MASK]` never appears at fine-tuning time; the
10% random and 10% unchanged cases force the encoder to build a good
representation of **every** token (it can't tell which positions will be scored),
reducing the pretrain/finetune mismatch.

### 3.4 Next-Sentence Prediction (conceptual)
A binary head over the pooled [CLS] representation
$h_{\text{CLS}}=\tanh(W\,h_0)$ predicts whether segment B truly follows A:
$\mathcal{L}_{\text{NSP}}=-\log p_\theta(\text{IsNext}\mid h_{\text{CLS}})$. The
total pretraining loss is $\mathcal{L}_{\text{MLM}}+\mathcal{L}_{\text{NSP}}$.
(Later work — RoBERTa — drops NSP.)

## 4. Key component — the MLM masking scheme + bidirectional attention

In [ ]:
# ===== actual implementation from bert.py =====
from __future__ import annotations

import math

import numpy as np

SEED = 0

PAD, CLS, SEP, MASK = 0, 1, 2, 3

N_SPECIAL = 4

def mlm_mask_numpy(tokens: np.ndarray, vocab: int, mask_id: int = MASK,
                   n_special: int = N_SPECIAL, p: float = 0.15,
                   seed: int = SEED):
    r"""
    BERT's 80/10/10 masking. Pick 15% of non-special positions; of those:
        - 80%  -> replace with [MASK]
        - 10%  -> replace with a random token
        - 10%  -> keep unchanged
    The model must predict the ORIGINAL token at every chosen position. The 10%
    "keep" / "random" splits stop the model from only ever seeing [MASK] at train
    time (which never appears at fine-tuning time) and force it to model every
    token, not just the masked slots.

    Returns (corrupted_tokens, labels) where labels = original id at chosen
    positions and -100 elsewhere (so cross-entropy ignores them).
    """
    rng = np.random.default_rng(seed)
    corrupted = tokens.copy()
    labels = np.full_like(tokens, -100)
    for b in range(tokens.shape[0]):
        for t in range(tokens.shape[1]):
            if tokens[b, t] < n_special:        # never mask [PAD]/[CLS]/[SEP]
                continue
            if rng.random() < p:
                labels[b, t] = tokens[b, t]      # remember the answer
                r = rng.random()
                if r < 0.8:
                    corrupted[b, t] = mask_id            # 80% -> [MASK]
                elif r < 0.9:
                    corrupted[b, t] = rng.integers(n_special, vocab)  # 10% random
                # else 10% -> leave unchanged
    return corrupted, labels

import torch

import torch.nn as nn

import torch.nn.functional as F

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class BidirectionalSelfAttention(nn.Module):
    """Multi-head self-attention with an optional *padding* mask (no causal mask)."""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, pad_mask: torch.Tensor | None = None):
        B, L, _ = x.shape
        qkv = self.qkv(x).view(B, L, 3, self.h, self.d_k).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]                       # (B, h, L, d_k)
        scores = q @ k.transpose(-1, -2) / math.sqrt(self.d_k)
        if pad_mask is not None:                               # (B, L): 1=keep, 0=pad
            scores = scores.masked_fill(pad_mask[:, None, None, :] == 0, float("-inf"))
        attn = self.drop(scores.softmax(-1))
        o = (attn @ v).transpose(1, 2).reshape(B, L, self.h * self.d_k)
        return self.out(o)

class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x): return self.net(x)

class EncoderBlock(nn.Module):
    """Pre-norm encoder block: x = x + Attn(LN(x)); x = x + FFN(LN(x))."""

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = BidirectionalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, pad_mask=None):
        x = x + self.attn(self.ln1(x), pad_mask)
        x = x + self.ff(self.ln2(x))
        return x

## 5. Full model — BERT (token/pos/segment emb, encoder, MLM + NSP heads)

In [ ]:
# ===== actual implementation from bert.py =====
class BERT(nn.Module):
    """Encoder-only Transformer with MLM head + pooled [CLS] head (NSP)."""

    def __init__(self, vocab: int, d_model: int = 64, n_heads: int = 4,
                 d_ff: int = 128, n_layers: int = 2, max_len: int = 64,
                 n_segments: int = 2, dropout: float = 0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, d_model, padding_idx=PAD)
        self.pos_emb = nn.Embedding(max_len, d_model)          # learned absolute
        self.seg_emb = nn.Embedding(n_segments, d_model)       # sentence A / B
        self.ln_in = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers))
        # MLM head (tied to input embedding) and a pooled NSP head.
        self.mlm = nn.Linear(d_model, vocab, bias=True)
        self.mlm.weight = self.tok_emb.weight                  # weight tying
        self.pool = nn.Linear(d_model, d_model)
        self.nsp = nn.Linear(d_model, 2)

    def encode(self, idx, seg=None, pad_mask=None):
        B, L = idx.shape
        pos = torch.arange(L, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None]
        if seg is not None:
            x = x + self.seg_emb(seg)
        x = self.drop(self.ln_in(x))
        for block in self.blocks:
            x = block(x, pad_mask)
        return x                                               # (B, L, d)

    def forward(self, idx, seg=None, pad_mask=None):
        h = self.encode(idx, seg, pad_mask)
        mlm_logits = self.mlm(h)                               # (B, L, vocab)
        pooled = torch.tanh(self.pool(h[:, 0]))                # [CLS] summary
        nsp_logits = self.nsp(pooled)                          # (B, 2)
        return mlm_logits, nsp_logits

def make_sequences(n: int, L: int, vocab: int, seed: int = SEED):
    """[CLS] w1 w2 ... wL [SEP]  with random content words (ids >= N_SPECIAL)."""
    rng = np.random.default_rng(seed)
    words = rng.integers(N_SPECIAL, vocab, size=(n, L))
    cls = np.full((n, 1), CLS)
    sep = np.full((n, 1), SEP)
    return np.concatenate([cls, words, sep], axis=1).astype(np.int64)

def demo():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    torch.set_num_threads(1)        # tiny CPU model: 1 thread avoids oversubscription
    dev = get_device()

    V, L, n = 24, 8, 384
    seqs = make_sequences(n, L, V)                             # (n, L+2)
    corrupt, labels = mlm_mask_numpy(seqs, V, p=0.20)          # mask ~20% to learn fast
    seg = np.zeros_like(seqs)                                  # single segment here
    pad_mask = (seqs != PAD).astype(np.int64)

    X = torch.tensor(corrupt, device=dev)
    Y = torch.tensor(labels, device=dev)
    S = torch.tensor(seg, device=dev)
    M = torch.tensor(pad_mask, device=dev)

    model = BERT(V, d_model=48, n_heads=4, d_ff=96, n_layers=2,
                 max_len=L + 2).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    lossfn = nn.CrossEntropyLoss(ignore_index=-100)

    model.train()
    for step in range(1, 251):
        mlm_logits, _ = model(X, S, M)
        loss = lossfn(mlm_logits.reshape(-1, V), Y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 50 == 0:
            # accuracy on the masked positions
            with torch.no_grad():
                pred = mlm_logits.argmax(-1)
                sel = Y != -100
                acc = (pred[sel] == Y[sel]).float().mean().item()
            print(f"step {step:4d}  loss {loss.item():.4f}  masked-acc {acc:.2f}")

    # show one example: original vs corrupted vs reconstruction at masked slots
    model.eval()
    with torch.no_grad():
        mlm_logits, _ = model(X[:1], S[:1], M[:1])
        pred = mlm_logits[0].argmax(-1)
    masked_pos = (Y[0] != -100).nonzero(as_tuple=True)[0].tolist()
    print("\noriginal   :", seqs[0].tolist())
    print("corrupted  :", X[0].tolist(), "  (id %d = [MASK])" % MASK)
    print("masked pos :", masked_pos)
    print("true @pos  :", [int(seqs[0][p]) for p in masked_pos])
    print("pred @pos  :", [int(pred[p]) for p in masked_pos])

## 6. Train / run — tiny masked-LM on toy `[CLS] … [SEP]` sequences

In [ ]:
demo()

## 7. Visualization — masking scheme & the MLM training loss curve

In [ ]:
import matplotlib
matplotlib.use("Agg")
import numpy as np, torch, matplotlib.pyplot as plt
import bert as M

torch.manual_seed(M.SEED); np.random.seed(M.SEED); torch.set_num_threads(1)

# (a) which positions get masked, on a small batch
V, L, n = 24, 8, 12
seqs = M.make_sequences(n, L, V)
corrupt, labels = M.mlm_mask_numpy(seqs, V, p=0.20)
chosen = (labels != -100).astype(float)          # 1 where a token is predicted

# (b) a short MLM training run for the loss curve
seqs_tr = M.make_sequences(384, L, V)
corr_tr, lab_tr = M.mlm_mask_numpy(seqs_tr, V, p=0.20, seed=1)
pad = (seqs_tr != M.PAD).astype(np.int64)
X = torch.tensor(corr_tr); Y = torch.tensor(lab_tr)
S = torch.zeros_like(X); P = torch.tensor(pad)
model = M.BERT(V, d_model=48, n_heads=4, d_ff=96, n_layers=2, max_len=L + 2)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
lossfn = torch.nn.CrossEntropyLoss(ignore_index=-100)
losses = []
for _ in range(120):
    mlm_logits, _ = model(X, S, P)
    loss = lossfn(mlm_logits.reshape(-1, V), Y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
im = ax[0].imshow(chosen, cmap="Reds", aspect="auto")
ax[0].set_title("Masked positions (red = predicted)")
ax[0].set_xlabel("position"); ax[0].set_ylabel("sequence")
ax[1].plot(losses); ax[1].set_xlabel("step"); ax[1].set_ylabel("MLM cross-entropy")
ax[1].set_title("Masked-LM loss going down")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- BERT = a Transformer **encoder** trained by **masked-token reconstruction**;
  bidirectional context is its whole point and the reason it can't be a generator.
- The 80/10/10 scheme is not cosmetic — it removes the train/finetune `[MASK]`
  mismatch and forces the model to represent every token.
- Only ~15% of tokens produce a loss signal, so MLM pretraining is sample-hungry
  compared to causal LM (every token supervises GPT).
- Pitfalls: masking special tokens ([CLS]/[SEP]/[PAD]); forgetting the padding
  mask (the model attends to pad slots); using BERT for generation (it has no
  causal structure).
- Compare: **[GPT](gpt.ipynb)** (causal, generative) and the full
  **[T5](t5.ipynb)** encoder-decoder.